### Handling of Spatial Variables

处理空间相关的变量

In [ ]:
import sys
sys.path.append('..')
from path_config import *

import myfunction as mf


import os
import pandas as pd
import numpy as np
import xarray as xr



import warnings
warnings.filterwarnings('ignore')


path_var_data = drive_letter + "/wyy/SPDB_database/data/raw/"


In [ ]:
# 函数准备

def count_variance(df_sem_data):
    """计算所有变量的方法  返回的是一个df"""
    print(df_sem_data.shape)
    mean = df_sem_data.mean()
    std_dev = df_sem_data.std()
    variance = df_sem_data.var()

    df_results = pd.DataFrame({
        'var': df_sem_data.columns,
        'mean': mean.values,
        'SD': std_dev.values,
        'variance': variance.values
    })

    min_variance = df_results["variance"].min()
    max_variance = df_results["variance"].max()
    print(min_variance, max_variance)
    print(max_variance/min_variance)
    return df_results

def convert_temp(df_o, from_scale, to_scale):
    """"转换温度变量的单位,C:Celsius,F:Fahrenheit,K:Kelvin,现在只支持覆盖全球的变量,非全球的0也会被变换"""
    df = df_o.copy()
    df_meta = pd.read_csv(path_file + meta_file, encoding="utf-8")
    list_all_var = df_meta['var_name'][(df_meta['var_select_' + 'all']==1)&(df_meta['var_temp']==1)].to_list()
    for col in list_all_var:
        if from_scale == "C":
            if to_scale == "F":
                df[col] = (df[col] * 9/5) + 32
            elif to_scale == "K":
                df[col] = df[col] + 273.15
        elif from_scale == "F":
            if to_scale == "C":
                df[col] = (df[col] - 32) * 5/9
            elif to_scale == "K":
                df[col] = ((df[col] - 32) * 5/9) + 273.15
        elif from_scale == "K":
            if to_scale == "C":
                df[col] = df[col] - 273.15
            elif to_scale == "F":
                df[col] = ((df[col] - 273.15) * 9/5) + 32
    return df




def dataset_to_dataframe(ds, var_name, index_vars):
    """
    Convert an xarray Dataset to a pandas DataFrame.
    
    :param ds: xarray Dataset to convert
    :param var_name: the variable name to extract from the Dataset
    :param index_vars: a list of variable names to use as index in the DataFrame
    :return: pandas DataFrame with the specified variable and index
    """
    df = ds[var_name].to_dataframe().reset_index()
    df = df.set_index(index_vars)
    return df

def process_nc_dir_one(nc_dir_one, list_var, df_geo, int_grid):
    for filename in os.listdir(nc_dir_one):
        if filename in list_var:
            with xr.open_dataset(os.path.join(nc_dir_one, filename)) as ds:
                var_name = [var for var in ds.data_vars][0]
                column_name = filename.replace("_" + str(int_grid) + "x" + str(int_grid) + ".nc", "")
                df_nc = dataset_to_dataframe(ds, var_name, ['lon', 'lat'])
                
                df_geo = df_geo.merge(df_nc, left_on=['lon_grid', 'lat_grid'], right_index=True, how='left')
                df_geo = df_geo.rename(columns={var_name: column_name})
    return df_geo

def process_nc_dir_year(nc_dir_year, list_var, df_geo, int_grid):
    for filename in os.listdir(nc_dir_year):
        if filename in list_var:
            with xr.open_dataset(os.path.join(nc_dir_year, filename)) as ds:
                ds = ds.sortby('lon')
                ds = ds.sortby('lat')
                ds = ds.sortby('year')
                var_name = [var for var in ds.data_vars][0]
                column_name = filename.replace("_" + str(int_grid) + "x" + str(int_grid) + ".nc", "")
                df_nc = dataset_to_dataframe(ds, var_name, ['lon', 'lat', 'year'])
                
                df_geo = df_geo.merge(df_nc, left_on=['lon_grid', 'lat_grid', 'year'], right_index=True, how='left')
                df_geo = df_geo.rename(columns={var_name: column_name})

    return df_geo



In [ ]:
# 一次性代码
# 生成
import pandas as pd
import numpy as np

lat_range = np.arange(-90, 90, 1)
lon_range = np.arange(-180, 180, 1)
year_range = np.arange(2000, 2051, 1)

df1 = pd.DataFrame({
    'lat_grid': np.repeat(lat_range, len(lon_range) * len(year_range)),
    'lon_grid': np.tile(np.repeat(lon_range, len(year_range)), len(lat_range)),
    'year': np.tile(year_range, len(lat_range) * len(lon_range))
})

df2 = pd.DataFrame({
    'lat_grid': np.repeat(lat_range, len(lon_range)),
    'lon_grid': np.tile(lon_range, len(lat_range))
})

print(df1.shape)
print(df2.shape)

df1.to_csv(path_part0_match + "lat_lon_year.csv", index=False)
df2.to_csv(path_part0_match + "lat_lon.csv", index=False)


In [ ]:
# ——————————————————————————————————————————————————————————————————
int_grid = 1
data_type = "raw"
# ——————————————————————————————————————————————————————————————————

nc_dir_year = path_var_data + str(int_grid) + "\\year"
nc_dir_one = path_var_data + str(int_grid) + "\\one"

df_meta = pd.read_csv(path_file + meta_file, encoding="utf-8")
list_var = df_meta['var_name'][(df_meta['file_type']=='nc')&(df_meta['var_select_all']==1)].tolist()
list_var = [var + "_" + str(int_grid) + "x" + str(int_grid) + ".nc" for var in list_var]
print(list_var)
print(len(list_var))

In [ ]:
# 分成21文件+非遍历高速匹配


for year in range(2000, 2051):
    print(year)
    df_geo = pd.read_csv(path_part0_match + "lat_lon.csv")
    df_geo = df_geo.drop_duplicates(subset=['lat_grid', 'lon_grid'])
    df_geo['year'] = year
    print(df_geo.columns)
    print(df_geo.shape)
    
    df_geo = process_nc_dir_one(nc_dir_one,list_var, df_geo, int_grid)
    df_geo = process_nc_dir_year(nc_dir_year,list_var, df_geo, int_grid)

    df_geo.fillna(0, inplace=True)

    def replace_nan_in_array(array):
        return np.where(np.isnan(array), 0, array)

    df_geo2 = df_geo.applymap(replace_nan_in_array)
    df_geo2 = df_geo2.fillna(0)
    df_geo2 = df_geo2.replace(["nan", "NaN", "NAN", "", None], 0)
    column_means = df_geo2.mean()

    filtered_columns = column_means[(column_means > 1) | (column_means == 0)]
    columns_list = filtered_columns.index.tolist()

    print(columns_list)
    col_drop = ['plev','plev_x','plev_y']
    df_geo2 = df_geo2.drop(columns=col_drop, errors='ignore')
    print(df_geo2.columns)
    df_geo2.to_csv(path_part0_geo + 'geo_global_'+str(year)+'.csv', encoding='utf-8-sig', index=False)

# 30 min

In [ ]:
# 合并匹配完成的21个csv文件
import os
import pandas as pd

file_names = [file for file in os.listdir(path_part0_geo) if file.endswith('.csv')]

dfs = [pd.read_csv(os.path.join(path_part0_geo, file)) for file in file_names]
merged_df = pd.concat(dfs, ignore_index=True)
merged_df.to_csv(path_part0_pre + 'geo_global_raw.csv', index=False)

In [ ]:
# 一次性
import mygeo as mg
df_geo_data = pd.read_csv(path_part0_pre + 'geo_global_raw.csv')
list_remove = ['lon_grid', 'lat_grid', 'year']

df_result, df_transformed = mg.transform_geo_data(
        df_geo_data=df_geo_data,
        list_remove=list_remove,
    )
df_result.to_csv(path_part0_temp + 'geo_transform_stats.csv', index=False)

df_result.to_pickle(path_part0_temp + "geo_transform_params.pkl")

df_transformed.to_csv(path_part0_temp + 'geo_variables_auto.csv', index=False)

df_geo_norm_z, scaler_geo_z = mg.normalize_geo_from_csv(df_transformed, 'zscore')

df_geo_norm_z.to_csv(path_part0_temp + 'geo_variables_auto_zscore.csv', index=False, float_format='%.9f')


### match

In [ ]:
def match_geo_data(df_lr, df_sw, df_geo, po_sheet, sp_sheet, treat_value=False):
    dict_inf_lr = {po_sheet:["po_carbon","po_f_carbon","po_chain","po_m_w","log_Px","log_Koc", "log_Kow","log_Kaw",
                        "log_Koa_wet","log_KHxd_air","log_Koil_w","log_Koil_air","density",
                        "solubility","log_pKa", 'log_D5_5', 'log_D7_4']
                        ,sp_sheet:["sp_length","sp_weight","sp_troph"]}
    dict_inf_sw = {po_sheet:["po_carbon","po_f_carbon","po_chain","po_m_w","log_Px","log_Koc", "log_Kow","log_Kaw",
                        "log_Koa_wet","log_KHxd_air","log_Koil_w","log_Koil_air","density",
                        "solubility","log_pKa", 'log_D5_5', 'log_D7_4']}
    df_lr_add = mf.append_inf(df_lr, dict_inf_lr, treat_value)
    df_sw_add = mf.append_inf(df_sw, dict_inf_sw, treat_value)

    df_lr_add = df_lr_add[df_lr_add['sp_length'].notna()]

    columns_to_drop = ['type']
    df_lr_add = df_lr_add.drop(columns=columns_to_drop)

    print(df_geo.columns)
    df_lr_all = pd.merge(df_geo, df_lr_add, on=['lat_grid', 'lon_grid', 'year'],how='right')
    df_sw_all = pd.merge(df_geo, df_sw_add, on=['lat_grid', 'lon_grid', 'year'],how='right')

    print(df_lr_all.isnull().sum()[df_lr_all.isnull().sum() > 0])
    print(df_sw_all.isnull().sum()[df_sw_all.isnull().sum() > 0])

    df_sw_all_copy = df_sw_all.copy()

    df_sw_all_copy.rename(columns={'value': 'sw_value'}, inplace=True)
    df_sw_all_copy = df_sw_all_copy[['lat_grid', 'lon_grid', 'posname', 'year', 'sw_value']]
    df_lr_sw = pd.merge(df_lr_all, df_sw_all_copy, on=['lat_grid', 'lon_grid', 'posname', 'year'], how='left')
    df_lr_sw = df_lr_sw[df_lr_sw['sw_value'].notna()]

    df_lr_full_analysis = df_lr_all.copy()
    df_lr_full_analysis = mf.select_data(df_lr_full_analysis, path_file, 'all')

    df_lr_analysis = df_lr_all.copy()
    df_sw_analysis = df_sw_all.rename(columns={'type': 'habitat'})
    df_lr_sw_analysis = df_lr_sw.copy()

    df_lr_analysis = mf.select_data(df_lr_analysis, path_file, 'bio')
    df_sw_analysis = mf.select_data(df_sw_analysis, path_file, 'w')
    df_lr_sw_analysis = mf.select_data(df_lr_sw_analysis, path_file, 'all')
    return df_lr_analysis, df_sw_analysis, df_lr_sw_analysis, df_lr_full_analysis

In [ ]:
df_lr_treat = pd.read_csv(path_part0_pre + "lr.csv")
print(df_lr_treat.columns)
df_sw_treat = pd.read_csv(path_part0_pre + "sw.csv")
print(df_sw_treat.columns)

df_geo_log_z = pd.read_csv(path_part0_temp + "geo_variables_auto_zscore.csv")
df_lr_analysis_z, df_sw_analysis_z, df_lr_sw_analysis_z, df_lr_full_analysis_z = match_geo_data(df_lr_treat, df_sw_treat, df_geo_log_z, 'posname', 'spid', treat_value=['log', 'zscore'])
df_lr_analysis_z.to_csv(path_part0_match + "lr_match_geo_auto_zscore.csv", index=False, float_format='%.9f')
df_sw_analysis_z.to_csv(path_part0_match + "sw_match_geo_auto_zscore.csv", index=False, float_format='%.9f')
df_lr_sw_analysis_z.to_csv(path_part0_match + "lr_sw_match_geo_auto_zscore.csv", index=False, float_format='%.9f')
df_lr_full_analysis_z.to_csv(path_part0_match + "lr_full_match_geo_auto_zscore.csv", index=False, float_format='%.9f')

df_geo_log = pd.read_csv(path_part0_temp + "geo_variables_auto.csv")
df_lr_analysis, df_sw_analysis, df_lr_sw_analysis, df_lr_full_analysis = match_geo_data(df_lr_treat, df_sw_treat, df_geo_log, 'posname', 'spid', treat_value=['log'])
df_lr_analysis.to_csv(path_part0_match + "lr_match_geo_auto.csv", index=False)
df_sw_analysis.to_csv(path_part0_match + "sw_match_geo_auto.csv", index=False)
df_lr_sw_analysis.to_csv(path_part0_match + "lr_sw_match_geo_auto.csv", index=False)
df_lr_full_analysis.to_csv(path_part0_match + "lr_full_match_geo_auto.csv", index=False)


df_geo_raw = pd.read_csv(path_part0_temp + "geo_variables_raw.csv")
df_lr_analysis, df_sw_analysis, df_lr_sw_analysis, df_lr_full_analysis = match_geo_data(df_lr_treat, df_sw_treat, df_geo_raw, 'posname', 'spid')
df_lr_analysis.to_csv(path_part0_match + "lr_match_geo_raw.csv", index=False)
df_sw_analysis.to_csv(path_part0_match + "sw_match_geo_raw.csv", index=False)
df_lr_sw_analysis.to_csv(path_part0_match + "lr_sw_match_geo_raw.csv", index=False)
df_lr_full_analysis.to_csv(path_part0_match + "lr_full_match_geo_raw.csv", index=False)



In [ ]:
# auto 转换沿用，zscore独立参数,
from scipy.special import boxcox as boxcox_with_lmbda
from scipy.stats import boxcox
df_sw_raw = pd.read_csv(path_part0_match + "sw_match_geo_auto.csv")
df_sw_raw['value'], lam_sw = boxcox(df_sw_raw['value'])


df_lr_raw = pd.read_csv(path_part0_match + "lr_match_geo_auto.csv")
df_lr_raw['value'], lam_lr = boxcox(df_lr_raw['value'])

df_lrsw_auto_m = pd.read_csv(path_part0_match + "lr_sw_match_geo_auto.csv")
df_lrsw_auto_m['value'] = boxcox_with_lmbda(df_lrsw_auto_m['value'], lam_lr)
df_lrsw_auto_m['sw_value'] = boxcox_with_lmbda(df_lrsw_auto_m['sw_value'], lam_sw)

from scipy.stats import zscore

cols_to_zscore = df_lrsw_auto_m.columns.difference(['lon_grid', 'lat_grid', 'year', 'habitat', 'organ_muscle', 
                                                     'organ_liver', 'po_chain', 'po_carbon', 'po_f_carbon', 'posname', 'spid'])

df_lrsw_auto_m[cols_to_zscore] = df_lrsw_auto_m[cols_to_zscore].apply(zscore)

df_lrsw_auto_m.to_csv(path_part0_match + "lr_sw_linear_z.csv", index=False)
df_lr_sw_treat = pd.read_csv(path_part0_match + "lr_sw_linear_z.csv")
# 对 lon, lat, year 生成 cluster
df_lr_sw_treat['cluster'] = df_lr_sw_treat.groupby(['lon_grid', 'lat_grid', 'year']).ngroup()
df_lr_sw_treat = df_lr_sw_treat.groupby('cluster').filter(lambda x: len(x) >= 10)
print(len(df_lr_sw_treat['cluster'].unique().tolist()))
print(df_lr_sw_treat['cluster'].value_counts())
df_lr_sw_treat.to_csv(path_part0_match + "lr_sw_linear_cluster_z.csv", index=False)